## Mixed continuous-discrete Bayesian optimization

This example demonstrates Bayesian optimization over a mixed design space with one continuous variable and one discrete variable.

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from xopt import VOCS, Xopt
from xopt.evaluator import Evaluator
from xopt.generators.bayesian import UpperConfidenceBoundGenerator

### Define a mixed-variable optimization problem

- `x_cont` is continuous on `[0, 1]`
- `x_disc` is discrete with allowed values `{0.0, 1.0, 2.0, 3.0}`

The objective has its minimum near `x_cont = 0.65` and `x_disc = 2.0`.

In [ ]:
def evaluate_mixed(input_dict):
    x_cont = float(input_dict["x_cont"])
    x_disc = float(input_dict["x_disc"])
    y = (x_cont - 0.65) ** 2 + (x_disc - 2.0) ** 2 + 0.05 * np.sin(6.0 * x_cont)
    return {"y": y}


vocs = VOCS(
    variables={"x_cont": [0.0, 1.0], "x_disc": {0.0, 1.0, 2.0, 3.0}},
    objectives={"y": "MINIMIZE"},
)


evaluator = Evaluator(function=evaluate_mixed)
generator = UpperConfidenceBoundGenerator(vocs=vocs)

# Keep the demo lightweight.
generator.n_monte_carlo_samples = 32
generator.numerical_optimizer.n_restarts = 4

torch.manual_seed(7)
np.random.seed(7)

X = Xopt(generator=generator, evaluator=evaluator)

### Seed the run with initial data

In [ ]:
initial_data = pd.DataFrame(
    {
        "x_cont": [0.10, 0.80, 0.30, 0.60, 0.20, 0.90],
        "x_disc": [0.0, 0.0, 1.0, 1.0, 2.0, 3.0],
    }
)

initial_evaluations = evaluator.evaluate_data(initial_data)
X.add_data(initial_evaluations)
X.data.tail()

### Run Bayesian optimization

Generate candidates from the Bayesian generator, evaluate them, and append the results.

In [ ]:
for _ in range(8):
    candidates = X.generator.generate(1)
    evaluations = evaluator.evaluate_data(candidates)
    X.add_data(evaluations)

X.data.tail()

In [ ]:
best_row = X.data.nsmallest(1, "y")[["x_cont", "x_disc", "y"]]
best_row

### Visualize samples by discrete mode

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for disc_value, group in X.data.groupby("x_disc"):
    ax.scatter(group["x_cont"], group["y"], label=f"x_disc={disc_value}")

ax.set_xlabel("x_cont")
ax.set_ylabel("objective y")
ax.set_title("Mixed continuous-discrete Bayesian optimization")
ax.legend(loc="best")
plt.show()